In [1]:
!pip install -q sentence-transformers

import numpy as np

def _cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

class SentenceEmbedder:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(model_name)
            self.use_transformer = True
        except Exception:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.vectorizer = TfidfVectorizer()
            self.use_transformer = False

    def encode(self, texts):
        if self.use_transformer:
            return self.model.encode(texts)
        else:
            return self.vectorizer.fit_transform(texts).toarray()

def find_most_similar(query, candidates, embedder):
    all_texts = [query] + candidates
    vectors = embedder.encode(all_texts)
    query_vec = vectors[0]
    best_score, best_match = -1, None
    for i, cand in enumerate(candidates):
        score = _cosine_similarity(query_vec, vectors[i + 1])
        if score > best_score:
            best_score, best_match = score, cand
    return best_match, best_score

In [2]:
feedback_a = 'Payment failed during checkout.'
feedback_b = 'Unable to complete my card transaction.'
feedback_c = 'The application interface looks beautiful.'

embedder = SentenceEmbedder()
vectors = embedder.encode([feedback_a, feedback_b, feedback_c])

print('A vs B similarity:', round(_cosine_similarity(vectors[0], vectors[1]), 3))
print('A vs C similarity:', round(_cosine_similarity(vectors[0], vectors[2]), 3))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

A vs B similarity: 0.588
A vs C similarity: -0.04


In [3]:
match, score = find_most_similar(feedback_a, [feedback_b, feedback_c], embedder)
print(f'Most similar to {feedback_a!r}: {match!r} (score={score:.3f})')

Most similar to 'Payment failed during checkout.': 'Unable to complete my card transaction.' (score=0.588)


In [4]:
try:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    text = 'The payment was not successful.'
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.convert_tokens_to_ids(tokens)

    print('Tokens:', tokens)
    print('Token IDs:', ids)
except Exception as e:
    print('transformers library not available in this environment:', e)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokens: ['the', 'payment', 'was', 'not', 'successful', '.']
Token IDs: [1996, 7909, 2001, 2025, 3144, 1012]


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
